# Notebook 1: Inspect the existing MongoDB database

Run this first. It connects to your database, lists collections, shows sample documents
and samples field types, so you can write the real schema into Notebook 2.

**Security:** use a read-only MongoDB account. Avoid printing confidential or personal data in a shared notebook.

## Cell 1: connect and list collections

In [ ]:
import os
from pprint import pprint
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()

MONGODB_URI = os.environ["MONGODB_URI"]
DATABASE_NAME = os.environ["MONGODB_DATABASE"]

client = MongoClient(
    MONGODB_URI,
    serverSelectionTimeoutMS=10000,
    connectTimeoutMS=10000
)

client.admin.command("ping")
db = client[DATABASE_NAME]

print("Connected successfully!")
print("Database:", DATABASE_NAME)
print("Collections:", db.list_collection_names())

## Cell 2: inspect a few example documents

Replace `students` with a real collection name.

In [ ]:
COLLECTION_NAME = "students"  # Replace this

collection = db[COLLECTION_NAME]

for document in collection.find({}, {"_id": 0}).limit(3):
    pprint(document)

## Cell 3: sample field types

This is sample-based inspection, not a guarantee that all documents share the same fields and types. Review the output and define the production schema manually.

In [ ]:
from collections import defaultdict
from bson import ObjectId
from datetime import datetime

def python_type_name(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int):
        return "integer"
    if isinstance(value, float):
        return "number"
    if isinstance(value, str):
        return "string"
    if isinstance(value, ObjectId):
        return "ObjectId"
    if isinstance(value, datetime):
        return "datetime"
    if isinstance(value, list):
        return "array"
    if isinstance(value, dict):
        return "object"
    return type(value).__name__


def inspect_schema(collection, sample_size=200):
    field_types = defaultdict(set)

    cursor = collection.find({}, {"_id": 0}).limit(sample_size)

    for document in cursor:
        for field, value in document.items():
            field_types[field].add(python_type_name(value))

    return {
        field: sorted(types)
        for field, types in sorted(field_types.items())
    }


schema = inspect_schema(collection)
pprint(schema)

## Next step

Note the collection names and field types above, then replace the sample `students` schema in `natural_language_mongodb.ipynb` (Cell 1) with your real schema.